### **03 — Alignment and coronary seed selection**
This notebook prepares coronary endpoints for Voronoi territory generation, processing one patient at a time. Check skeleton against anatomical meshes before selecting roots. After tree separation, inspect the newly numbered endpoints and record exclusions using those updated numbers.


In [1]:
import os
from collections import deque
import numpy as np
import nibabel as nib
import pyvista as pv

path = "/Users/ahthini/Documents/KCL/individual project/imageCAS_30_samples"
heart_path = os.path.join(path, "10064282") #input patient ID number here
mesh_path = os.path.join(heart_path, "myocardium.stl")
rv_mesh_path = os.path.join(heart_path, "heart", "right_ventricle.stl")
skeleton16_path = os.path.join(heart_path, "heart", "label16mesh.vtk")
cleaned_mesh_path = os.path.join(heart_path, "myocardium_cleaned.vtp")
img_path = os.path.join(heart_path, "img.nii")
skeleton_path = os.path.join(heart_path, "label_skeleton.nii")
corrected_skeleton_path = os.path.join(heart_path, "label_skeleton_aligned.nii")

#load meshes
mesh = pv.read(mesh_path)
rv_mesh = pv.read(rv_mesh_path)
label16 = pv.read(skeleton16_path)

#mesh info
def print_mesh_info(name, mesh):
    print(f"\n--- {name} Mesh Info ---")
    print(f"Number of points: {mesh.n_points}")
    print(f"Number of cells: {mesh.n_cells}")
    print(f"Bounds: {mesh.bounds}")
    print(f"Center: {mesh.center}")
    print("Point data arrays:", mesh.point_data.keys())
    print("Cell data arrays:", mesh.cell_data.keys())

print_mesh_info("Myocardium", mesh)
print_mesh_info("Right Ventricle", rv_mesh)
print_mesh_info("Label 16", label16)


--- Myocardium Mesh Info ---
Number of points: 249170
Number of cells: 498340
Bounds: (-89.25589752197266, -22.082000732421875, 116.9749984741211, 190.51600646972656, 120.19999694824219, 193.1999969482422)
Center: [-55.668949127197266, 153.74550247192383, 156.6999969482422]
Point data arrays: []
Cell data arrays: []

--- Right Ventricle Mesh Info ---
Number of points: 145548
Number of cells: 291092
Bounds: (-64.29728698730469, 7.525390625, 143.91766357421875, 190.833984375, 117.19999694824219, 203.96484375)
Center: [-28.385948181152344, 167.37582397460938, 160.5824203491211]
Point data arrays: []
Cell data arrays: []

--- Label 16 Mesh Info ---
Number of points: 83042
Number of cells: 31324
Bounds: (-94.03119659423828, 12.9375, 117.03500366210938, 194.33599853515625, 121.93499755859375, 206.13099670410156)
Center: [-40.54684829711914, 155.6855010986328, 164.03299713134766]
Point data arrays: ['normals']
Cell data arrays: []


**Prepare skeleton's spatial coordinates**

The original code applies the CT image affine to the skeleton volume and saves it as label_skeleton_aligned.nii. Skeleton voxel indices are then converted to world coordinates. The centre of the skeleton's bounding box is calculated for the visualisation.

The centre translation used later for plotting is applied to coordinate arrays only; it is not written into the saved NIfTI file. The saved affine and displayed translation are therefore separate operations in the existing implementation.

In [2]:
#load and align skeleton
ref_img = nib.load(img_path)
skeleton_img = nib.load(skeleton_path)
skeleton_data = skeleton_img.get_fdata().astype(np.uint8)
fixed_skeleton = nib.Nifti1Image(skeleton_data, affine=ref_img.affine, header=skeleton_img.header)
fixed_skeleton.set_qform(ref_img.affine)
fixed_skeleton.set_sform(ref_img.affine)
nib.save(fixed_skeleton, corrected_skeleton_path)
skeleton_img = nib.load(corrected_skeleton_path)
skeleton_data = skeleton_img.get_fdata().astype(np.uint8)
affine = skeleton_img.affine

#get skeleton in world coordinates
skeleton_vox = np.argwhere(skeleton_data > 0)
skeleton_world = nib.affines.apply_affine(affine, skeleton_vox)
skel_min = skeleton_world.min(axis=0)
skel_max = skeleton_world.max(axis=0)
skel_center = (skel_min + skel_max) / 2
print("\n--- Skeleton Bounds ---")
print(f"x: {skel_min[0]:.3f} to {skel_max[0]:.3f}")
print(f"y: {skel_min[1]:.3f} to {skel_max[1]:.3f}")
print(f"z: {skel_min[2]:.3f} to {skel_max[2]:.3f}")
print("Skeleton center:", skel_center)


--- Skeleton Bounds ---
x: -92.599 to 10.868
y: 118.407 to 192.903
z: 124.450 to 204.450
Skeleton center: [-40.86523438 155.65527344 164.44999695]


**Detect coronary endpoints**

An endpoint is a skeleton voxel with exactly one neighbouring foreground voxel in its 26-neighbourhood. The code prints a numbered endpoint list containing:
- voxel indices
- corresponding world coordinates

Keep the voxel indices when recording anatomical landmarks.
Endpoint numbers may change after left-right separation.

In [3]:
#find endpoints
neighbors_26 = np.array([[x, y, z] for x in [-1, 0, 1]
                         for y in [-1, 0, 1]
                         for z in [-1, 0, 1]
                         if (x, y, z) != (0, 0, 0)])

endpoints_coords = []
shape = skeleton_data.shape
for coord in skeleton_vox:
    count = 0
    for offset in neighbors_26:
        neighbor = coord + offset
        if (0 <= neighbor[0] < shape[0] and
            0 <= neighbor[1] < shape[1] and
            0 <= neighbor[2] < shape[2]):
            if skeleton_data[tuple(neighbor)] > 0:
                count += 1
    if count == 1:
        endpoints_coords.append(coord)

endpoints_coords = np.array(endpoints_coords)
endpoints_world = nib.affines.apply_affine(affine, endpoints_coords)
print(f"\nTotal endpoints found: {len(endpoints_coords)}")
print("--- Endpoint Summary ---")
for i, (vox, world) in enumerate(zip(endpoints_coords, endpoints_world)):
    vox_str = f"Voxel: [{vox[0]}, {vox[1]}, {vox[2]}]"
    world_str = f"World: ({world[0]:.2f}, {world[1]:.2f}, {world[2]:.2f})"
    print(f"Seed {i+1:02d} | {vox_str:<25} | {world_str}")


Total endpoints found: 16
--- Endpoint Summary ---
Seed 01 | Voxel: [179, 317, 80]     | World: (-13.65, 182.08, 136.95)
Seed 02 | Voxel: [192, 320, 76]     | World: (-17.78, 183.03, 134.95)
Seed 03 | Voxel: [201, 277, 197]    | World: (-20.65, 169.34, 195.45)
Seed 04 | Voxel: [210, 162, 94]     | World: (-23.51, 132.73, 143.95)
Seed 05 | Voxel: [217, 184, 59]     | World: (-25.74, 139.74, 126.45)
Seed 06 | Voxel: [241, 167, 64]     | World: (-33.38, 134.33, 128.95)
Seed 07 | Voxel: [258, 207, 211]    | World: (-38.80, 147.06, 202.45)
Seed 08 | Voxel: [261, 191, 198]    | World: (-39.75, 141.97, 195.95)
Seed 09 | Voxel: [271, 343, 148]    | World: (-42.93, 190.36, 170.95)
Seed 10 | Voxel: [307, 161, 67]     | World: (-54.40, 132.42, 130.45)
Seed 11 | Voxel: [313, 140, 175]    | World: (-56.31, 125.73, 184.45)
Seed 12 | Voxel: [326, 117, 146]    | World: (-60.44, 118.41, 169.95)
Seed 13 | Voxel: [329, 342, 55]     | World: (-61.40, 190.04, 124.45)
Seed 14 | Voxel: [415, 294, 104]    | 

**Inspect the skeleton and anatomical surfaces**

For visualisation, the skeleton and endpoint coordinates are translated so the skeleton's bounding-box centre matches the coronary mesh centre. The plot overlays the myocardium, right ventricle, coronary surface, skeleton, and numbered endpoints.

Rotate the view and inspect:
- the correspondence between the skeleton and coronary surface;
- the left and right coronary roots;
- clustered or apparently artificial endpoints.

Record root voxel indices before continuing.

In [5]:
#translate skeleton and endpoints
label16_center = label16.center
skeleton_translated = skeleton_world - skel_center + label16_center
endpoints_translated = endpoints_world - skel_center + label16_center
print("\n--- Alignment Check ---")
print("Skeleton center:", skel_center)
print("Label16 mesh center:", label16_center)
print("Translation vector:", label16_center - skel_center)
cleaned_myo = pv.read(cleaned_mesh_path) #load cleaned myocardium

#endpoint labels
voxel_labels = [f"{i+1}: [{v[0]}, {v[1]}, {v[2]}]" for i, v in enumerate(endpoints_coords)]
pv.set_jupyter_backend("trame")
plotter = pv.Plotter(shape=(1, 1), window_size=(1200, 1000)) #visualization

#left: anatomy
#plotter.subplot(0, 0)
subject_id = os.path.basename(heart_path)
plotter.add_text(f"ID {subject_id} | Skeleton + Myocardium + RV + Label16", font_size=10)
plotter.add_mesh(mesh, color="lightgray", opacity=0.7)
plotter.add_mesh(rv_mesh, color="skyblue", opacity=0.7)
plotter.add_mesh(label16, color="red", opacity=0.5)
plotter.add_points(skeleton_translated, color="red", point_size=2, render_points_as_spheres=True)
plotter.add_points(endpoints_translated, color="yellow", point_size=6, render_points_as_spheres=True)
plotter.add_point_labels(endpoints_translated, voxel_labels,
                         font_size=20, show_points=False,
                         always_visible=True, fill_shape=True,
                         shape_opacity=0.25, text_color="black")

#right: cleaned myocardium only
"""plotter.subplot(0, 1)
plotter.add_text("Myocardium Cleaned Mesh", font_size=10)
plotter.add_mesh(cleaned_myo, color="white", opacity=0.4)
plotter.add_points(endpoints_translated, color="red", point_size=6, render_points_as_spheres=True)
plotter.add_point_labels(endpoints_translated, labels,
                         font_size=10, show_points=False,
                         always_visible=True, fill_shape=True,
                         shape_opacity=0.25, text_color="black")

plotter.link_views()"""
plotter.view_isometric()
plotter.show()


--- Alignment Check ---
Skeleton center: [-40.86523438 155.65527344 164.44999695]
Label16 mesh center: [-40.54684829711914, 155.6855010986328, 164.03299713134766]
Translation vector: [ 0.31838608  0.03022766 -0.41699982]


Widget(value='<iframe src="http://localhost:54723/index.html?ui=P_0x124f930e0_1&reconnect=auto" class="pyvista…

 JS Error => TypeError: Cannot destructure property 'topic' of 'e' as it is undefined.


**Specify left and right coronary roots**

Enter the voxel indices of the roots identified for this patient.

When the optional seed arrays are empty, separation starts from
the two roots. For difficult cases, manually identified seeds
can be supplied for each side. For each side, a non-empty seed array replaces that side's root as the starting seed set in the existing algorithm.

In [11]:
show_vis = True
skeleton_path = os.path.join(heart_path, "label_skeleton_aligned.nii")
left_out_path = os.path.join(heart_path, "skeleton_left.nii")

#roots
left_root_voxel  = np.array([258, 207, 211], dtype=int) #change this
right_root_voxel = np.array([201, 277, 197], dtype=int) #change this

#manually identified LEFT & RIGHT seeds (from table) - voxel indices
left_seed_voxels = np.array([], dtype=int)
right_seed_voxels= np.array([], dtype=int)

**Prepare the skeleton neighbourhood and starting seeds**

The saved skeleton is loaded and its foreground voxels identified. Neighbour relationships use 26-connectivity. Supplied root or seed coordinates are snapped to the nearest skeleton voxel when they do not already lie on the skeleton. The starting seeds are then prepared for left-right separation.

In [12]:
#load skeleton
img = nib.load(skeleton_path)
skel = img.get_fdata().astype(np.uint8)
affine = img.affine
shape = skel.shape
if skel.max() == 0:
    raise RuntimeError("Skeleton volume has no foreground voxels.")

#neighborhood (26-connectivity)
neighbors_26 = np.array( [[x, y, z] for x in (-1, 0, 1)
               for y in (-1, 0, 1)
               for z in (-1, 0, 1)
               if not (x == 0 and y == 0 and z == 0)], dtype=int)

def in_bounds(v):
    return (0 <= v[0] < shape[0]) and (0 <= v[1] < shape[1]) and (0 <= v[2] < shape[2])

#precompute foreground voxel list for snapping
skel_vox_list = np.argwhere(skel > 0)
def snap_to_skeleton_voxel(vox):
    vi = tuple(int(x) for x in vox)
    if in_bounds(vi) and skel[vi] > 0:
        return np.array(vi, dtype=int)
    d2 = np.sum((skel_vox_list - np.asarray(vox)[None, :])**2, axis=1)
    return skel_vox_list[int(np.argmin(d2))].astype(int)

def find_endpoints():
    #Endpoints = degree 1 in 26-nhood (within skeleton)
    ep = []
    for (x, y, z) in skel_vox_list:
        cnt = 0
        for off in neighbors_26:
            nx, ny, nz = x+off[0], y+off[1], z+off[2]
            if 0 <= nx < shape[0] and 0 <= ny < shape[1] and 0 <= nz < shape[2]:
                if skel[nx, ny, nz] > 0:
                    cnt += 1
        if cnt == 1:
            ep.append([x, y, z])
    return np.array(ep, dtype=int)

#prepare seed sets
def snap_list(arr):
    return np.array([snap_to_skeleton_voxel(v) for v in arr], dtype=int) if arr.size else np.empty((0,3), int)

#if a side’s seed list is provided -> use it; otherwise use that side’s root voxel.
left_seeds = snap_list(left_seed_voxels) if left_seed_voxels.size \
             else np.array([snap_to_skeleton_voxel(left_root_voxel)], dtype=int)

right_seeds = snap_list(right_seed_voxels) if right_seed_voxels.size \
              else np.array([snap_to_skeleton_voxel(right_root_voxel)], dtype=int)

print(f"Inputted seeds -> left: {left_seeds.shape[0]} seeds | right: {right_seeds.shape[0]} seeds")

Inputted seeds -> left: 1 seeds | right: 1 seeds


**Separate the left and right coronary trees**

A traversal expands from the left and right starting seeds through the skeleton. Skeleton voxels disconnected from both starting seed sets are
assigned using their distance to the nearest left or right seed
in voxel coordinates.

The left coronary mask is saved as skeleton_left.nii. Both masks remain available in memory for visual inspection.

In [13]:
#competitive BFS (graph Voronoi) - label map: 0 = unlabeled, 1 = left, 2 = right
labelled = np.zeros(shape, dtype=np.uint8)
q = deque()

def enqueue_seed(v, lbl):
    v = tuple(int(x) for x in v)
    if not in_bounds(v): 
        return
    if skel[v] == 0:
        return
    if labelled[v] != 0:
        return
    labelled[v] = lbl
    q.append(v)

#seed both sides
for v in left_seeds: enqueue_seed(v, 1)
for v in right_seeds: enqueue_seed(v, 2)
while q:
    cx, cy, cz = q.popleft()
    cur_lbl = labelled[cx, cy, cz]
    for off in neighbors_26:
        nx, ny, nz = cx + off[0], cy + off[1], cz + off[2]
        if not (0 <= nx < shape[0] and 0 <= ny < shape[1] and 0 <= nz < shape[2]):
            continue
        if skel[nx, ny, nz] == 0:
            continue
        if labelled[nx, ny, nz] != 0:
            continue
        labelled[nx, ny, nz] = cur_lbl
        q.append((nx, ny, nz))

#any leftover skeleton (disconnected from seeds) -> assign to the closer side (in voxel L2)
left_mask  = (labelled == 1)
right_mask = (labelled == 2)
unlab = (skel > 0) & (labelled == 0)
if np.any(unlab):
    ul = np.argwhere(unlab)
    # if right seeds empty for some reason, treat all leftovers as right
    if right_seeds.size == 0:
        labelled[unlab] = 2
    else:
        # decide by nearest seed set (voxel L2) – light fallback
        for v in ul:
            dl = np.min(np.linalg.norm(left_seeds  - v, axis=1)) if left_seeds.size  else np.inf
            dr = np.min(np.linalg.norm(right_seeds - v, axis=1)) if right_seeds.size else np.inf
            labelled[tuple(v)] = 1 if dl <= dr else 2
    left_mask  = (labelled == 1)
    right_mask = (labelled == 2)

#save outputs
nib.save(nib.Nifti1Image(left_mask.astype(np.uint8),  affine), left_out_path)
print(f"[✓] Left-only mask   -> {left_out_path}")

[✓] Left-only mask   -> /Users/ahthini/Documents/KCL/individual project/imageCAS_30_samples/10064282/skeleton_left.nii


**Identify and inspect the left coronary endpoints**

Endpoints are detected again on the separated left coronary skeleton and assigned a new sequence of seed numbers. The plot displays the two coronary trees and labels the left endpoints.

Check that:
- the branches have been assigned to the appropriate side
- the left root is identifiable
- clustered or artificial endpoints are noted
- any new endpoint introduced by the split is reviewed

Record exclusions using the new left-only seed numbers and their voxel indices. Exclusions are applied during Voronoi seed selection in notebook 04.

In [14]:
#endpoints on the LEFT-ONLY skeleton
left_vox_list = np.argwhere(left_mask > 0)
def left_is_endpoint(ixyz):
    x, y, z = ixyz
    deg = 0
    for off in neighbors_26:
        nx, ny, nz = x + off[0], y + off[1], z + off[2]
        if 0 <= nx < shape[0] and 0 <= ny < shape[1] and 0 <= nz < shape[2]:
            if left_mask[nx, ny, nz] > 0:
                deg += 1
                if deg > 1:
                    return False
    return deg == 1

left_endpoints_vox = []
for v in left_vox_list:
    if left_is_endpoint(v):
        left_endpoints_vox.append(v.tolist())
left_endpoints_vox = np.array(left_endpoints_vox, dtype=int)

print(f"\n[Left-only] total endpoints found: {len(left_endpoints_vox)}")
print("--- Left endpoints (voxel indices) ---")
for i, v in enumerate(left_endpoints_vox, start=1):
    print(f"Seed {i:02d} | Voxel: [{v[0]}, {v[1]}, {v[2]}]")

#visualiation
if show_vis:
    left_vox  = np.argwhere(left_mask)
    right_vox = np.argwhere(right_mask)
    lw = nib.affines.apply_affine(affine, left_vox)  if left_vox.size  else np.empty((0,3))
    rw = nib.affines.apply_affine(affine, right_vox) if right_vox.size else np.empty((0,3))

    # Transform left endpoints into world coords
    left_endpoints_world = nib.affines.apply_affine(affine, left_endpoints_vox) if left_endpoints_vox.size else np.empty((0,3))

    plotter = pv.Plotter(window_size=(1200, 850))
    if rw.size: 
        plotter.add_points(rw, color="red",  point_size=5, render_points_as_spheres=True)
    if lw.size: 
        plotter.add_points(lw, color="blue", point_size=6, render_points_as_spheres=True)
    if left_endpoints_world.size:
        label_texts = [f"Seed {i:02d}\nVoxel {v.tolist()}"
                    for i, v in enumerate(left_endpoints_vox, start=1)]
        plotter.add_point_labels( left_endpoints_world, label_texts, font_size=16,
            text_color="black", always_visible=True, shape="rounded_rect",
            shape_opacity=0.85, margin=0, render_points_as_spheres=False)

    #make labels stay a constant screen size & frame the view
    plotter.enable_parallel_projection() 
    plotter.view_isometric()
    plotter.camera.zoom(1.4)

    plotter.add_legend([("Left Coronary Tree", "blue"), 
        ("Right Coronary Tree", "red"), 
        ("Left Endpoints", "yellow")])
    subject_id = os.path.basename(heart_path)
    plotter.add_text(
        f"ID {subject_id} | Skeletons: Left (Blue), Right (Red), Endpoints (Yellow)",
        font_size=16)
    plotter.show()


[Left-only] total endpoints found: 8
--- Left endpoints (voxel indices) ---
Seed 01 | Voxel: [258, 207, 211]
Seed 02 | Voxel: [261, 191, 198]
Seed 03 | Voxel: [313, 140, 175]
Seed 04 | Voxel: [326, 117, 146]
Seed 05 | Voxel: [329, 342, 55]
Seed 06 | Voxel: [415, 294, 104]
Seed 07 | Voxel: [415, 295, 104]
Seed 08 | Voxel: [420, 264, 106]


Widget(value='<iframe src="http://localhost:54723/index.html?ui=P_0x123576570_2&reconnect=auto" class="pyvista…

**Complete this record after inspecting the plots.**

| Item | Value |
|---|---|
| Patient ID | |
| Left root voxel indices | |
| Right root voxel indices | |
| Additional left starting seeds, if used | |
| Additional right starting seeds, if used | |
| Left-only seed numbers to exclude | |
| Voxel indices of excluded seeds | |
| Reasons for exclusion | |
| Alignment or separation issues | |

Use the same patient ID and the exclusions recorded above for notebook 4.